# 02 — Circuit comparison across the 4 configs
baseline_orig (apricot+orig) · m3_stale (apricot+edited) ·
m1_scratch (scratch+edited) · m2-* (fine-tuned+edited).
RUN IN THE CIRCUIT-TRACER ENV (clts/.venv-ct).

In [ ]:
# --- setup + load manifest ---
import sys, json
from pathlib import Path
REPO = Path.cwd()
while REPO.name != "Interp_LM4" and REPO != REPO.parent: REPO = REPO.parent
sys.path.insert(0, str(REPO))
from clts.storage import storage_root
from clts.edit_clt import edit_clt_config as cfg, compare, drift
STORE = storage_root()
EXP_ID = "grid-L4-H6-edit-p0-month-jul"
M = cfg.read_manifest(STORE / "edit_experiments" / EXP_ID / "manifest.json")
PROMPT = M["trace_prompt"]; DATA_DIR = M["data_dir"]
GRAPH_ROOT = STORE / "clt_graphs"
print("prompt:", PROMPT)
print("apricot target ce_recovered:", round(M["target_stats"]["ce_recovered"], 4))

In [ ]:
# --- build (or reuse) the 4 graphs on the same prompt ---
gcs = compare.configs_from_manifest(M)
results = {}
for gc in gcs:
    if not Path(gc.clt_dir).exists():
        print("SKIP (missing CLT):", gc.key, gc.clt_dir); continue
    print("building:", gc.key)
    results[gc.key] = compare.build_or_load_graph(
        gc, DATA_DIR, GRAPH_ROOT, slug=EXP_ID, prompt=PROMPT, device="cpu")
reports = {k: r["report"] for k, r in results.items()}
graphs = {k: r["graph"] for k, r in results.items()}

In [ ]:
# --- replacement-model stats across configs (the headline comparison) ---
compare.comparison_table(reports)

In [ ]:
# --- also trace the SAME target tokens (old vs new month) across configs ---
for tgt in (" February", " July"):
    print("target:", tgt)
    for gc in gcs:
        if gc.key not in graphs: continue
        r = compare.build_or_load_graph(gc, DATA_DIR, GRAPH_ROOT,
                slug=f"{EXP_ID}{tgt.strip()}", prompt=PROMPT, target=tgt, device="cpu")
        print(f"  {gc.key:14s} repl={r['report']['replacement_score']:.3f} "
              f"err={r['report']['error_influence_share']:.3f}")

In [ ]:
# --- feature diff vs baseline: what appears / disappears ---
compare.feature_diff_table(graphs, baseline_key="baseline_orig")

In [ ]:
# --- weight-space drift (M2/M3 share apricot indexing; M1 needs matching) ---
from clts.clt import CrossLayerTranscoder
base_clt = CrossLayerTranscoder.load_from_dir(M["base_clt_dir"])
for key, m in M["methods"].items():
    d = Path(m["expected_clt_dir"])
    if not d.exists(): continue
    other = CrossLayerTranscoder.load_from_dir(d)
    if key.startswith("m2"):
        dr = drift.decoder_cosine_drift(base_clt, other)
        print(f"{key}: mean_cos={dr['mean_cosine']:.4f} frac_moved={dr['frac_moved']:.3f}")
    else:  # m1 from scratch -> per-layer best-match cosine
        cos0 = drift.match_features(base_clt, other, layer=0)["match_cosine"].mean()
        print(f"{key}: L0 mean best-match cos to apricot = {float(cos0):.3f}")

In [ ]:
# --- Method-3 focus: did unexplained influence rise vs baseline? ---
if "baseline_orig" in reports and "m3_stale" in reports:
    b, s = reports["baseline_orig"], reports["m3_stale"]
    print(f"error_influence_share: baseline={b['error_influence_share']:.3f} "
          f"-> m3_stale={s['error_influence_share']:.3f} "
          f"(Δ={s['error_influence_share']-b['error_influence_share']:+.3f})")
    ov = drift.active_feature_overlap(graphs["baseline_orig"], graphs["m3_stale"])
    print("features lost when stale CLT meets the edit:", ov["disappeared_set"])

In [ ]:
# --- save a comparison report (table + diffs) ---
out = STORE / "edit_experiments" / EXP_ID / "comparison.json"
out.write_text(json.dumps({"reports": reports,
    "feature_diff": compare.feature_diff_table(graphs).to_dict(orient="index")},
    indent=2, default=str))
print("wrote", out)

## Optional: interactive viewer
`clts/.venv-ct/bin/python clts/serve_ui.py --graph-dir <clt_graphs/<key>/<EXP_ID>> --scan-name <key> --port 8050`